# 02 Ergebnis-Analyse

Dieses Notebook liest die erzeugten CSVs ein und zeigt die wichtigsten Qualitaetskennzahlen.

In [1]:
from pathlib import Path
import json
import pandas as pd
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

IN_DIR = PROJECT_ROOT / 'artifacts' / 'real_data_mvp'
ANALYSIS_DIR = IN_DIR / 'analysis'
IN_DIR, ANALYSIS_DIR

(PosixPath('/home/felicitasl/Documents/rl_project/rl-fundamental-trading/artifacts/real_data_mvp'),
 PosixPath('/home/felicitasl/Documents/rl_project/rl-fundamental-trading/artifacts/real_data_mvp/analysis'))

In [2]:
prices = pd.read_csv(IN_DIR / 'prices_daily.csv', parse_dates=['date'])
fundamentals = pd.read_csv(IN_DIR / 'fundamentals_quarterly.csv', parse_dates=['period_end', 'report_date'])
features = pd.read_csv(IN_DIR / 'features_daily.csv', parse_dates=['date', 'report_date'])

len(prices), len(fundamentals), len(features)

(2256, 45, 2256)

In [ ]:
summary_path = ANALYSIS_DIR / 'dataset_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    summary
else:
    print('Noch kein Summary vorhanden. Erst run_analysis ausführen.')

In [ ]:
coverage_path = ANALYSIS_DIR / 'features_coverage_by_symbol.csv'
if coverage_path.exists():
    coverage = pd.read_csv(coverage_path)
    coverage
else:
    print('Noch keine Coverage-Datei vorhanden. Erst run_analysis ausführen.')

In [7]:
print("prices.head()")
display(prices.head())

print("fundamentals.head()")
display(fundamentals.head())

print("features.head()")
display(features.head())

prices.head()


,date,symbol,open,high,low,close,volume
0,2022-01-03,AAPL,173.703383,178.636198,173.586172,177.786377,104487900
1,2022-01-04,AAPL,178.392046,178.694850,174.963487,175.530029,99310400
2,2022-01-05,AAPL,175.442085,175.989088,170.587415,170.860916,94537600
3,2022-01-06,AAPL,168.692420,171.232092,167.657020,168.008667,96904000
4,2022-01-07,AAPL,168.878052,170.099046,167.061213,168.174759,86709100


fundamentals.head()


,symbol,period_end,report_date,report_type,filing_lag_days,revenue,net_income,operating_cashflow,debt_to_equity,gross_margin,roe
0,AAPL,2022-03-26,2022-04-29,quarterly,34,2.212230e+11,5.964000e+10,7.513200e+10,4.202777,0.437577,0.884880
1,AAPL,2022-06-25,2022-07-29,quarterly,34,3.041820e+11,7.908200e+10,9.802400e+10,4.787754,0.436209,1.360972
2,AAPL,2022-09-24,2022-10-28,annual,34,3.943280e+11,9.980300e+10,1.221510e+11,5.961537,0.433096,1.969589
3,AAPL,2022-09-24,2023-02-03,quarterly,132,NaN,NaN,NaN,5.961537,NaN,NaN
4,AAPL,2022-12-31,2023-02-03,quarterly,34,1.171540e+11,2.999800e+10,3.400500e+10,5.112557,0.429623,0.528813


features.head()


,date,symbol,open,high,low,close,volume,period_end,report_date,report_type,filing_lag_days,revenue,net_income,operating_cashflow,debt_to_equity,gross_margin,roe,is_report_day,days_since_report,has_fundamentals
0,2022-01-03,AAPL,173.703383,178.636198,173.586172,177.786377,104487900,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,999,0
1,2022-01-04,AAPL,178.392046,178.694850,174.963487,175.530029,99310400,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,999,0
2,2022-01-05,AAPL,175.442085,175.989088,170.587415,170.860916,94537600,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,999,0
3,2022-01-06,AAPL,168.692420,171.232092,167.657020,168.008667,96904000,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,999,0
4,2022-01-07,AAPL,168.878052,170.099046,167.061213,168.174759,86709100,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,999,0


In [5]:
# Schnelle Mindestchecks fuer Modell-Tauglichkeit
lookahead = ((features['report_date'].notna()) & (features['report_date'] > features['date'])).sum()
coverage_ratio = float(features['has_fundamentals'].mean())

print({'lookahead_violations': int(lookahead), 'with_fundamentals_ratio': coverage_ratio})

{'lookahead_violations': 0, 'with_fundamentals_ratio': 0.9064716312056738}


## Erweiterte Datenanalyse

Im Folgenden kommen typische QA-Checks und domänenspezifische Finanzdaten-Analysen für die Modellvorbereitung.

In [8]:
import numpy as np

def missingness_table(df, name):
    miss = df.isna().mean().sort_values(ascending=False).rename('missing_ratio')
    out = miss.to_frame()
    out['dataset'] = name
    out['dtype'] = [str(df[c].dtype) for c in out.index]
    return out.reset_index(names='column')

missing_prices = missingness_table(prices, 'prices')
missing_fund = missingness_table(fundamentals, 'fundamentals')
missing_feat = missingness_table(features, 'features')

print('Top Missingness - prices')
display(missing_prices.head(10))

print('Top Missingness - fundamentals')
display(missing_fund.head(15))

print('Top Missingness - features')
display(missing_feat.head(15))

Top Missingness - prices


,column,missing_ratio,dataset,dtype
0,date,0.0,prices,datetime64[us]
1,symbol,0.0,prices,str
2,open,0.0,prices,float64
3,high,0.0,prices,float64
4,low,0.0,prices,float64
5,close,0.0,prices,float64
6,volume,0.0,prices,int64


Top Missingness - fundamentals


,column,missing_ratio,dataset,dtype
0,gross_margin,0.2,fundamentals,float64
1,revenue,0.2,fundamentals,float64
2,net_income,0.2,fundamentals,float64
3,roe,0.2,fundamentals,float64
4,operating_cashflow,0.2,fundamentals,float64
5,symbol,0.0,fundamentals,str
6,period_end,0.0,fundamentals,datetime64[us]
7,report_date,0.0,fundamentals,datetime64[us]
8,filing_lag_days,0.0,fundamentals,int64
9,report_type,0.0,fundamentals,str


Top Missingness - features


,column,missing_ratio,dataset,dtype
0,report_date,0.093528,features,datetime64[us]
1,report_type,0.093528,features,str
2,filing_lag_days,0.093528,features,float64
3,period_end,0.093528,features,str
4,roe,0.093528,features,float64
5,gross_margin,0.093528,features,float64
6,operating_cashflow,0.093528,features,float64
7,net_income,0.093528,features,float64
8,revenue,0.093528,features,float64
9,debt_to_equity,0.093528,features,float64


In [9]:
# Basis-Qualitaetschecks: Duplikate, Zeitabdeckung, Lags
dup_prices = int(prices.duplicated(subset=['symbol', 'date']).sum())
dup_fund = int(fundamentals.duplicated(subset=['symbol', 'period_end', 'report_date', 'report_type']).sum())

coverage_by_symbol = (
    prices.groupby('symbol', as_index=False)
    .agg(
        n_days=('date', 'size'),
        start=('date', 'min'),
        end=('date', 'max')
    )
    .sort_values('symbol')
)

lag_stats = fundamentals['filing_lag_days'].describe(percentiles=[0.1, 0.5, 0.9, 0.95]).to_frame(name='filing_lag_days')
negative_lag_rows = int((fundamentals['filing_lag_days'] < 0).sum())

print({'duplicate_prices_rows': dup_prices, 'duplicate_fund_rows': dup_fund, 'negative_filing_lag_rows': negative_lag_rows})
print('Price coverage by symbol')
display(coverage_by_symbol)
print('Filing lag distribution')
display(lag_stats)

{'duplicate_prices_rows': 0, 'duplicate_fund_rows': 0, 'negative_filing_lag_rows': 0}
Price coverage by symbol


,symbol,n_days,start,end
0,AAPL,752,2022-01-03,2024-12-30
1,MSFT,752,2022-01-03,2024-12-30
2,NVDA,752,2022-01-03,2024-12-30


Filing lag distribution


,filing_lag_days
count,45.000000
mean,48.022222
std,37.483626
min,19.000000
10%,24.000000
50%,31.000000
90%,120.000000
95%,124.400000
max,132.000000


In [10]:
# Event-Analyse: Verhalten an Report-Tagen vs normalen Tagen
feat_evt = features.copy()
feat_evt = feat_evt.sort_values(['symbol', 'date']).reset_index(drop=True)
feat_evt['ret_1d'] = feat_evt.groupby('symbol')['close'].pct_change()

event_stats = (
    feat_evt.groupby('is_report_day')['ret_1d']
    .agg(['count', 'mean', 'std', 'median'])
    .rename(index={0: 'non_report_day', 1: 'report_day'})
)

# Volatilitaet in +/- 3 Handelstagen um Report
event_window = feat_evt[feat_evt['days_since_report'].between(0, 3, inclusive='both')].copy()
non_event_window = feat_evt[feat_evt['days_since_report'] > 20].copy()

window_compare = {
    'event_window_rows': int(len(event_window)),
    'non_event_window_rows': int(len(non_event_window)),
    'event_window_abs_ret_mean': float(event_window['ret_1d'].abs().mean()),
    'non_event_window_abs_ret_mean': float(non_event_window['ret_1d'].abs().mean()),
}

display(event_stats)
print(window_compare)

,count,mean,std,median
is_report_day,,,,
non_report_day,2219,0.001209,0.024443,0.001269
report_day,34,0.003401,0.031059,-0.003970


{'event_window_rows': 89, 'non_event_window_rows': 1758, 'event_window_abs_ret_mean': 0.0248248920175579, 'non_event_window_abs_ret_mean': 0.016837644680010717}


In [11]:
# Fundamentals-Plausibilitaet: einfache Bereichschecks
plaus = {
    'gross_margin_outside_[0,1]': int(((features['gross_margin'] < 0) | (features['gross_margin'] > 1)).sum()),
    'roe_outside_[-2,2]': int(((features['roe'] < -2) | (features['roe'] > 2)).sum()),
    'debt_to_equity_gt_20': int((features['debt_to_equity'] > 20).sum()),
    'revenue_non_positive_with_fundamentals': int(((features['has_fundamentals'] == 1) & (features['revenue'] <= 0)).sum()),
}

quantiles = features[['revenue', 'net_income', 'operating_cashflow', 'debt_to_equity', 'gross_margin', 'roe']].quantile([0.01, 0.05, 0.5, 0.95, 0.99])
print(plaus)
display(quantiles)

{'gross_margin_outside_[0,1]': 0, 'roe_outside_[-2,2]': 0, 'debt_to_equity_gt_20': 0, 'revenue_non_positive_with_fundamentals': 0}


,revenue,net_income,operating_cashflow,debt_to_equity,gross_margin,roe
0.01,7.192000e+09,1.618000e+09,1.731000e+09,0.456972,0.429623,0.061474
0.05,8.288000e+09,2.043000e+09,2.911000e+09,0.529341,0.433096,0.083320
0.50,1.185370e+11,3.398100e+10,3.989500e+10,0.990608,0.655285,0.350887
0.95,3.910350e+11,9.699500e+10,1.185480e+11,5.408780,0.766333,1.645935
0.99,3.943280e+11,9.980300e+10,1.221510e+11,5.961537,0.783520,1.969589


In [12]:
# Sehr einfacher Signal-Check fuer Modellintuition
signal = features.copy().sort_values(['symbol', 'date']).reset_index(drop=True)
signal['fwd_ret_5d'] = signal.groupby('symbol')['close'].pct_change(5).shift(-5)

valid = signal[signal['has_fundamentals'] == 1].copy()
valid = valid.dropna(subset=['fwd_ret_5d', 'roe', 'gross_margin', 'debt_to_equity'])

corrs = {
    'corr_roe_vs_fwd_ret_5d': float(valid['roe'].corr(valid['fwd_ret_5d'])) if len(valid) else np.nan,
    'corr_gross_margin_vs_fwd_ret_5d': float(valid['gross_margin'].corr(valid['fwd_ret_5d'])) if len(valid) else np.nan,
    'corr_debt_to_equity_vs_fwd_ret_5d': float(valid['debt_to_equity'].corr(valid['fwd_ret_5d'])) if len(valid) else np.nan,
    'n_valid_rows': int(len(valid)),
}
print(corrs)

{'corr_roe_vs_fwd_ret_5d': -0.029853936749308512, 'corr_gross_margin_vs_fwd_ret_5d': 0.020500715340382183, 'corr_debt_to_equity_vs_fwd_ret_5d': -0.048941148221172404, 'n_valid_rows': 2030}
